In [ ]:
# Colab setup (uncomment if running in Google Colab)
# !pip install -q pandas numpy lightgbm xgboost optuna scikit-learn
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/sentinel-poc/project_sentinel

# Model Training: Cascaded Two-Stage Architecture

This notebook trains the cascaded Stage 1 and Stage 2 models for the Early Warning System.

## Architecture
- **Stage 1 (High-Recall Filter)**: Uses high-frequency, continuously available vital signs and readily available demographics to flag at-risk patients early. Designed to maximize recall and generate a dynamic risk score.
- **Stage 2 (High-Precision Classifier)**: For patients flagged by Stage 1, incorporates slower-turnaround laboratory values (e.g., Creatinine, BUN) and comprehensive data to confirm risk. Designed to maximize precision and reduce alarm fatigue.

In [ ]:
import os
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import optuna

from src import models

# Define paths
PROCESSED_DATA_DIR = '../data/processed'
MODELS_DIR = '../models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Load labeled splits
print("Loading labeled splits...")
X_train = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'X_train.parquet'))
y_train = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'y_train.parquet'))
X_val = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'X_val.parquet'))
y_val = pd.read_parquet(os.path.join(PROCESSED_DATA_DIR, 'y_val.parquet'))

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

In [ ]:
print("Training Stage 1 Models (Vitals only)...")
stage1_models = models.train_stage1_models(X_train, y_train, X_val, y_val)
print("Stage 1 Models trained successfully.")

In [ ]:
print("Training Stage 2 Models (LightGBM + XGBoost with Optuna tuning)...")
stage2_models = models.train_stage2_models(X_train, y_train, X_val, y_val)
print("Stage 2 Models trained successfully.")

In [ ]:
print("Saving all models to disk...")
models.save_models(stage1_models, stage2_models, MODELS_DIR)
print("Models saved successfully.")

## Training Summary

We have successfully trained the cascaded models:
- **Stage 1 Models** are built on continuously available features (Vitals + Demographics).
- **Stage 2 Models** utilize the full feature set including laboratory results, tuned using Optuna.
- All models are persisted in the `models/` directory for evaluation and subsequent deployment.